In [292]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import explained_variance_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import classification_report, accuracy_score
import math

# Loading mast data

We chose to go with the Risoe data, only loading the data that is needed, to reduce memory usage.

In [ ]:
import netCDF4 as nc
import numpy as np
from datetime import datetime, timedelta

file_path_risoe = 'Data/Risoe/risoe_m_all.nc'
file_paths_borglum = 'Data/Borglum/borglum_all.nc'

signals_risoe = ['ws77', 'wd77', 'ws125', 'wd125']
signals_borglum = ['ws32', 'wd32']

base_date_borglum = datetime(1997, 12, 11, 16, 5, 0)
base_date_risoe = datetime(1995, 11, 20, 16, 25, 0)

# Get the Risoe dataset:
dataset = nc.Dataset(file_path_risoe, 'r')

# List the variables in the dataset
print("Variables in the netCDF file:")
for var_name in dataset.variables:
    print(var_name)

time_minutes = np.array(dataset.variables['time'])

# Convert time values to timestamp strings
time = []
for minutes in time_minutes:
	time_delta = timedelta(minutes=int(minutes))
	timestamp = base_date_risoe + time_delta
	time.append(timestamp.strftime('%Y-%m-%d %H:%M:%S'))
 
print(f"time:\n {time[:10]} - {time[-1]}")

for signal in signals_risoe:
	values = np.array(dataset.variables[signal])
	print(f'{signal}:\n {values[:10]} - {values[-10:-1]}')

In [ ]:
import netCDF4 as nc
import numpy as np
import pandas as pd

# Make sure to use the correct dataset
file_path = 'Data/Risoe/risoe_m_all.nc'
dataset = nc.Dataset(file_path, 'r')

# Confirm variables are in the dataset
print("Variables in the netCDF file:")
for var_name in dataset.variables:
    print(var_name)

# Ensure the required variables exist before attempting to access them
required_vars = ['time', 'ws77', 'wd77', 'ws125', 'wd125']
for var in required_vars:
    if var not in dataset.variables:
        print(f"Variable {var} is missing in the dataset.")
    else:
        print(f"Variable {var} is available.")

# Convert NetCDF data to pandas DataFrame
df_mast_risoe = pd.DataFrame()
df_mast_risoe['time'] = pd.to_datetime([datetime(1995, 11, 20, 16, 25) + timedelta(minutes=int(m)) for m in dataset.variables['time'][:]])
df_mast_risoe['ws77'] = np.array(dataset.variables['ws77'][:])
df_mast_risoe['wd77'] = np.array(dataset.variables['wd77'][:]%360)
df_mast_risoe['ws125'] = np.array(dataset.variables['ws125'][:])
df_mast_risoe['wd125'] = np.array(dataset.variables['wd125'][:]%360)

# Setting the 'time' column as the index
df_mast_risoe.set_index('time', inplace=True)

# Display the last 25 rows to confirm data is loaded correctly
print(df_mast_risoe.tail(25))


making windroses to determine the wind speeds from directions. count based

Converting the data to a Pandas DataFrame, to make it easier to work with.
Converting time to datetime objects, and setting it as the index.

In [295]:
df_mast_risoe = pd.DataFrame()
df_mast_risoe['time'] = pd.to_datetime([datetime(1995, 11, 20, 16, 25) + timedelta(minutes=int(m)) for m in dataset.variables['time'][:]])
df_mast_risoe['ws77'] = np.array(dataset.variables['ws77'][:])
df_mast_risoe['wd77'] = np.array(dataset.variables['wd77'][:])
df_mast_risoe['ws125'] = np.array(dataset.variables['ws125'][:])
df_mast_risoe['wd125'] = np.array(dataset.variables['wd125'][:])

df_mast_risoe.set_index('time', inplace=True)

## Handleing missing values

Plotting the data so se where data is mussing

In [ ]:
sns.set()
df_mast_risoe.plot(subplots=True, layout=(4,1), figsize=(12,12), sharex=False, sharey=False)

We chose to use height 77 to have more data points to work with.


In [ ]:
df_mast_risoe.drop(inplace=True, columns=['wd125', 'ws125'])
df_mast_risoe.info()

In [ ]:
df_mast_risoe['wd77'].value_counts().sort_index().plot(kind='bar', figsize=(12,4), grid=False)

In [299]:

#Remove the last part of data where the wind direction is missing i.e. 0.0 for a long period of time

series = pd.Series(df_mast_risoe['wd77'])
non_zero = series.to_numpy().nonzero()
last_non_zero = non_zero[0][-1]

# remove all values after last non-zero value
df_mast_risoe = df_mast_risoe.iloc[:last_non_zero+1]

Doing a visual inspection of the data again to see results after removing missing values.

In [ ]:
sns.set()
df_mast_risoe.plot(subplots=True, layout=(4,1), figsize=(12,12), sharex=False, sharey=False)

In [ ]:
# plot the first 6 months of ws77 from the start of 2004

df_mast_risoe['2004-01-01':'2004-07-01']['ws77'].plot(figsize=(12,4), grid=True)

In [ ]:
#Trying to display the missing data in a different way. To see if there were a lot of 0.0 measurements, but it is a lot of 0.2 which might still be valid.

df_mast_risoe['2004-01-01':'2004-07-01']['wd77'].value_counts().sort_index().plot(kind='bar', figsize=(12,4), grid=False)

In [ ]:
# Showing amount of null values again

df_mast_risoe.info()

In [ ]:

#Creating placeholder columns for dates to make it possible to do the transformation

df_mast_risoe['date_month_day_hour'] = df_mast_risoe.index.strftime('%m-%d %H')
df_mast_risoe['minute_first_digit'] = df_mast_risoe.index.minute // 10
df_mast_risoe['ws77'].fillna(df_mast_risoe.groupby(['date_month_day_hour', 'minute_first_digit'])['ws77'].transform('mean'), inplace=True)

In [ ]:
#After the transformation we remove the help columns again
df_mast_risoe.drop(['date_month_day_hour', 'minute_first_digit'], axis=1, inplace=True)



In [ ]:


df_mast_risoe['2004-01-01':'2004-07-01']['ws77'].plot(figsize=(12,4), grid=True)

In [ ]:
df_mast_risoe.info()

In [ ]:
from windrose import WindroseAxes
import matplotlib.pyplot as plt

ws_risoe_77 = df_mast_risoe['ws77'].to_numpy()
wd_risoe_77 = df_mast_risoe['wd77'].to_numpy()

# Plot wind rose
ax = WindroseAxes.from_ax()
ax.bar(wd_risoe_77, ws_risoe_77)
ax.set_legend()

plt.show()

In [ ]:
# Plot wind rose
ax = WindroseAxes.from_ax()
ax.bar(wd_risoe_77, ws_risoe_77, normed=True)
ax.set_legend()

# Format radius axis to percentages
fmt = '%.0f%%'
yticks = mtick.FormatStrFormatter(fmt)
ax.yaxis.set_major_formatter(yticks)

plt.show()

In [ ]:
# calculate ouliers

q1 = df_mast_risoe['ws77'].quantile(0.25)
q3 = df_mast_risoe['ws77'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - (1.5 * iqr)
upper_bound = q3 + (1.5 * iqr)
ws77_outlier_count = df_mast_risoe[(df_mast_risoe['ws77'] < lower_bound) | (df_mast_risoe['ws77'] > upper_bound)].count()['ws77']
print(f'ws77 outlier count: {ws77_outlier_count}')

In [ ]:
# boxplot to see outliers

sns.set()
sns.boxplot(x=df_mast_risoe['ws77'])

We chose to keep the outliers since wind speed does vary a lot and it is not a measurement error.
Plotting a histogram to visualize the distribution of the data. It looks close to a Wibull-distribution.

In [ ]:
# show histogram of df_mast_risoe

df_mast_risoe.hist(bins=50, figsize=(12,4))

Converting Mast time to UTC


In [ ]:
import pytz

# Reset index 
df_mast_risoe = df_mast_risoe.reset_index()

# Convert 'time' column to datetime format
df_mast_risoe['time'] = pd.to_datetime(df_mast_risoe['time'])

# Set timezone for Denmark and convert to UTC if 'time' is timezone-naive
# Made the if statement due to an error where the data was alreaddy transformed if this was run seperatly, else all code needed to be compiled again.
dk_time = pytz.timezone('Europe/Copenhagen')
if df_mast_risoe['time'].dt.tz is None:
    df_mast_risoe['time'] = df_mast_risoe['time'].dt.tz_localize(dk_time, nonexistent='shift_forward', ambiguous='NaT').dt.tz_convert('UTC')
else:
    df_mast_risoe['time'] = df_mast_risoe['time'].dt.tz_convert('UTC')

# Remove rows with any NaT values in 'time' after timezone conversion
df_mast_risoe = df_mast_risoe.dropna(subset=['time'])

# Set 'time' back as index if required
df_mast_risoe = df_mast_risoe.set_index('time')

print(df_mast_risoe)

In [ ]:
# Checking if null values are added in the conversion

df_mast_risoe.info()

Adding the missing WD data

In [ ]:
# find index of nan values in wd77

nan_index = df_mast_risoe[df_mast_risoe['wd77'].isnull()].index

nan_index

In [ ]:
# plot before adding missing values ws77 and wd77 from 2006-02-27 07:20 to 2006-02-27 08:05

df_mast_risoe['2006-02-27 05:00':'2006-02-27 09:00'].plot(subplots=True, layout=(4,1), figsize=(12,12), sharex=False, sharey=False)

In [317]:
# adding missing WD values by linear interopolate

df_mast_risoe['wd77'] = df_mast_risoe['wd77'].interpolate(method='linear')

In [ ]:
# Plotting again to se the full dataset

df_mast_risoe['2006-02-27 05:00':'2006-02-27 09:00'].plot(subplots=True, layout=(4,1), figsize=(12,12), sharex=False, sharey=False)

In [ ]:
# Scaling the data since linear regresion works best on scaled data

scaled_df_mast_risoe = pd.DataFrame(MinMaxScaler().fit_transform(df_mast_risoe), columns=df_mast_risoe.columns, index=df_mast_risoe.index)

scaled_df_mast_risoe.hist(bins=50, figsize=(12,4))

In [ ]:
# First we resample wind speed into a new dataframe column.

resampled_scaled_mast_risoe = pd.DataFrame()

# resample the ws77 data to 1 hour intervals 
resampled_scaled_mast_risoe['ws77'] = scaled_df_mast_risoe['ws77'].resample('1H').mean()


resampled_scaled_mast_risoe.dropna(inplace=True)
resampled_scaled_mast_risoe.head()

In [ ]:
# plot resampled_scaled_mast_risoe to verify it looks similar to the plot before

resampled_scaled_mast_risoe.plot(subplots=True, layout=(1,1), figsize=(12,4), sharex=False, sharey=False)

Centering allows us to treat the average more intuitively because we can now think of all angles as being measured from a "central" point.
For instance, if we have angles of 350° and 10°:
After shifting:
350 deg becomes 170° (350 + 180 = 530; 530 % 360 = 170)
10deg  becomes 190° (10 + 180 = 190)

after this we shift the average back by subtracting 180° to return it to the original scale.
In our case, 180 deg becomes 0 deg after shifting back (180 - 180 = 0).

lets say the mean is 170 deg (170 -180 = -10) then we take mod 360 = 350. and now its refered to the original value of 350 deg


In [ ]:
# Resample ws77 (wind speed) to hourly intervals
resampled_scaled_mast_risoe['ws77'] = scaled_df_mast_risoe['ws77'].resample('1H').mean()

# For wd77 (wind direction), use the circular mean approach without vectorization
def circular_mean(degrees_series):
    # Shift angles by 180 degrees to center them around the average
    shifted_degrees = (degrees_series + 180) % 360
    # Calculate the mean of the shifted angles and then shift back
    mean_direction = (shifted_degrees.mean() - 180) % 360
    return mean_direction

resampled_scaled_mast_risoe['wd77'] = scaled_df_mast_risoe['wd77'].resample('1H').apply(circular_mean)

In [ ]:
# Verifying no errors are added in the dataset 

resampled_scaled_mast_risoe.info()

In [ ]:
# plot resampled_scaled_mast_risoe

resampled_scaled_mast_risoe.plot(subplots=True, figsize=(12,4), sharex=False, sharey=False)

In [ ]:
# MESO DATA Begins here.
# only load TIMESTAMP WSP060 WSP080 WDIR060 WDIR080
# convert TIMESTAMP to datetime and use it as index

meso_risoe = pd.read_csv('Data/Risoe/meso_Risoe.csv', usecols=['TIMESTAMP', 'WSP060', 'WSP080', 'WDIR060', 'WDIR080'], parse_dates=['TIMESTAMP'], index_col='TIMESTAMP')


# sort dataframe by index
meso_risoe.sort_index(inplace=True)

meso_risoe.head()

In [ ]:
# plot meso_risoe
meso_risoe.plot(subplots=True, figsize=(12,4), sharex=False, sharey=False)

## Only using overlapping time period


In [ ]:
# The meso data ain't ordered by time but by index

resampled_scaled_mast_risoe.index[0], resampled_scaled_mast_risoe.index[-1]

In [ ]:
# To se the time span of the Meso data

meso_risoe.index[0], meso_risoe.index[-1]

In [ ]:
# Ensure meso_risoe has a timezone-aware index in UTC
meso_risoe.index = meso_risoe.index.tz_localize('UTC')

# Get the last index of resampled_scaled_mast_risoe
last_index = resampled_scaled_mast_risoe.index[-1]

# Slice meso_risoe to only include up to the last index of resampled_scaled_mast_risoe
meso_risoe_overlap = meso_risoe[:last_index]

# Plot overlap
meso_risoe_overlap.plot(subplots=True, figsize=(12, 4), sharex=False, sharey=False)

In [ ]:
# Convert meso_risoe index to UTC if it's already timezone-aware
meso_risoe.index = meso_risoe.index.tz_convert('UTC')

# Get the first index of meso_risoe
first_index = meso_risoe.index[0]

# Slice resampled_scaled_mast_risoe from the first index of meso_risoe
resampled_scaled_mast_risoe_overlap = resampled_scaled_mast_risoe[first_index:]

# Plot the overlap
resampled_scaled_mast_risoe_overlap.plot(subplots=True, figsize=(12, 4), sharex=False, sharey=False)


# Interpolate the wind speed height

In [ ]:
# interpolate the height to 77 for the MESO data. The plotting the difference between the interpolated WSP077 and the original WSP080

target_height = 77

# Calculate the interpolation weight for WS060 and WS080
w_WS060 = (80 - target_height) / (80 - 60)
w_WS080 = (target_height - 60) / (80 - 60)

# Linearly interpolate WS060 and WS080 to WS077
meso_risoe_overlap['WS077'] = (meso_risoe_overlap['WSP060'] * w_WS060) + (meso_risoe_overlap['WSP080'] * w_WS080)

# plot WS060 WS080 and WS077
meso_risoe_overlap[['WSP060', 'WSP080', 'WS077']].plot(subplots=True, figsize=(12,4), sharex=False, sharey=False)

In [ ]:
print (meso_risoe)

In [ ]:
# plot the difference between WSP080 and WS077
(meso_risoe_overlap['WSP080'] - meso_risoe_overlap['WS077']).plot(figsize=(12,4), grid=True)

In [ ]:
# Interpolation target height
target_height = 77

# calculate the interpolated wind direction as an angle weighted average of WDIR060 and WDIR080
# first convert the angles to radians
# then calculate the interpolation weight for WDIR060 and WDIR080
# then calculate the angle weighted average of WDIR060 and WDIR080

meso_risoe_overlap['WDIR060'] = np.radians(meso_risoe_overlap['WDIR060'])
meso_risoe_overlap['WDIR080'] = np.radians(meso_risoe_overlap['WDIR080'])

w_WDIR060 = (80 - target_height) / (80 - 60)
w_WDIR080 = (target_height - 60) / (80 - 60)

meso_risoe_overlap['WDIR077'] = (meso_risoe_overlap['WDIR060'] * w_WDIR060) + (meso_risoe_overlap['WDIR080'] * w_WDIR080)

# convret from radians to degrees

meso_risoe_overlap['WDIR077'] = np.degrees(meso_risoe_overlap['WDIR077'])

In [ ]:
meso_risoe_overlap[['WDIR060', 'WDIR080', 'WDIR077']].plot(subplots=True, figsize=(12,4), sharex=False, sharey=False)

In [ ]:
# plot the difference bewteen WDIR060 and WDIR080  

(meso_risoe_overlap['WDIR060'] - meso_risoe_overlap['WDIR080']).plot(figsize=(12,4), grid=True)

In [ ]:
# Checking for corelesson between meso and mast data

# join the two dataframes together

joined_df = pd.concat([resampled_scaled_mast_risoe_overlap, meso_risoe_overlap[['WS077', 'WDIR077']]], axis=1)

joined_df.dropna(inplace=True)

joined_df.head()

In [ ]:
joined_df.info()

In [ ]:
# plot joined_df

joined_df.plot(subplots=True, figsize=(12,4), sharex=False, sharey=False)

In [ ]:
#  correlation matrix plot

# Set up the plot style
sns.set(style="white")

# Calculate the correlation matrix
corr = joined_df.corr()

# Set up the mask to only show one triangle of the matrix
mask = np.triu(np.ones_like(corr, dtype=bool))

# Create a larger figure for better visibility of all features
plt.figure(figsize=(14, 12))

# Draw the heatmap
sns.heatmap(corr, mask=mask, cmap='coolwarm', vmax=1, vmin=-1, center=0,
            annot=True, square=True, linewidths=0.5, cbar_kws={"shrink": 0.75})

# Display the plot
plt.show()


# Distribution


In [ ]:
# plot histogram

joined_df.hist(bins=50, figsize=(12,4))

this looks like a Weibull distribution

more info https://en.wikipedia.org/wiki/Weibull_distribution

In [ ]:
from scipy.stats import weibull_min

# Fit a Weibull distribution to the data
params = weibull_min.fit(joined_df['ws77'])

# Create a histogram of the data
plt.hist(joined_df['ws77'], bins=30, density=True, alpha=0.6, label='Data Histogram')

# Create a range of x values for the Weibull PDF
x = np.linspace(0, joined_df['ws77'].max(), 100)

# Calculate the Weibull PDF using the fitted parameters
pdf = weibull_min.pdf(x, *params)

# Plot the Weibull PDF on top of the histogram
plt.plot(x, pdf, 'r-', lw=2, label='Weibull PDF')

# Add labels and a legend
plt.xlabel('ws77')
plt.ylabel('Probability Density')
plt.legend()

# Show the plot
plt.show()

In [ ]:
# Fit a Weibull distribution to the data
params = weibull_min.fit(joined_df['WS077'])

# Create a histogram of the data
plt.hist(joined_df['WS077'], bins=30, density=True, alpha=0.6, label='Data Histogram')

# Create a range of x values for the Weibull PDF
x = np.linspace(0, joined_df['WS077'].max(), 100)

# Calculate the Weibull PDF using the fitted parameters
pdf = weibull_min.pdf(x, *params)

# Plot the Weibull PDF on top of the histogram
plt.plot(x, pdf, 'r-', lw=2, label='Weibull PDF')

# Add labels and a legend
plt.xlabel('WS077')
plt.ylabel('Probability Density')
plt.legend()

# Show the plot
plt.show()

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


y = joined_df['WS077']              
X = joined_df.drop(['WS077'], axis=1)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define a range of alpha values to search
alpha_values = np.logspace(-4, 6, 1000)

# Create and fit the LassoCV model with tweaked parameters
lasso_cv_model = LassoCV(
    alphas=alpha_values,
    cv=10,
    fit_intercept=True,
    max_iter=8000,  # Increase max iterations
    tol=1e-5,       # Set a smaller tolerance
    random_state=42
)

lasso_cv_model.fit(X_train, y_train)

y_pred = lasso_cv_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error for Lasso: {mse}')

# Print the optimal alpha found by LassoCV
print(f'Optimal alpha found by LassoCV: {lasso_cv_model.alpha_}')

# Make predictions and calculate mean squared error
y_pred_cv = lasso_cv_model.predict(X_test)
mse_cv = mean_squared_error(y_test, y_pred_cv)
print(f'Mean Squared Error with Optimal Alpha from LassoCV: {mse_cv}')

print ('-----------------------------------------------------------------------------')
# Calculate and print R-squared for training and testing data
r2_train = lasso_cv_model.score(X_train, y_train)
r2_test = lasso_cv_model.score(X_test, y_test)
print(f'R-squared on training data is {r2_train}')  
print(f'R-squared on test data is {r2_test}')  

# Calculate Bias
bias = np.mean(y_pred - y_test)
print(f'Bias of LassoCV model: {bias}')

# Calculate Standard Deviation of Predictions
std_dev_predictions = np.std(y_pred)
print(f'Standard Deviation of Predictions for LassoCV: {std_dev_predictions}')




In [ ]:
# Plotting Actual vs. Predicted Values
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.3)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=4) 
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs. Predicted')
plt.grid(True)
plt.show()

In [ ]:
# A coeficent analasys
nfeatures = len(lasso_cv_model.coef_)
plt.figure(dpi = 800)
plt.barh(range(nfeatures), lasso_cv_model.coef_, align='center')
plt.yticks(np.arange(nfeatures), X.columns)
plt.xlabel("Feature importance")
plt.ylabel("Feature")

In [ ]:
print (joined_df)

In [ ]:
# Trying a ridgeModel

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


# Create and fit the ridgeCV model with tweaked parameters
ridge_cv_model = RidgeCV(
    alphas=alpha_values,
    cv=5,
    fit_intercept=True,
)

ridge_cv_model.fit(X_train, y_train)

y_pred_ridge = ridge_cv_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred_ridge)
print(f'Mean Squared Error for RidgeCV: {mse}')

# Print the optimal alpha found by LassoCV
print(f'Optimal alpha found by RidgeCV: {ridge_cv_model.alpha_}')

# Make predictions and calculate mean squared error
y_pred_cv = ridge_cv_model.predict(X_test)
mse_cv = mean_squared_error(y_test, y_pred_cv)
print(f'Mean Squared Error with Optimal Alpha from RidgeCV: {mse_cv}')


print ('-----------------------------------------------------------------------------')
# Calculate and print R-squared for training and testing data
r2_train = ridge_cv_model.score(X_train, y_train)
r2_test =  ridge_cv_model.score(X_test, y_test)
print(f'R-squared on training data is {r2_train}')  
print(f'R-squared on test data is {r2_test}')  

# Calculate Bias
bias = np.mean(y_pred_ridge - y_test)
print(f'Bias of RidgeCV model: {bias}')

# Calculate Standard Deviation of Predictions
std_dev_predictions = np.std(y_pred_ridge)
print(f'Standard Deviation of Predictions for RidgeCV: {std_dev_predictions}')




In [ ]:
# Plotting Actual vs. Predicted Values
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_ridge, alpha=0.3)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=4) 
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs. Predicted')
plt.grid(True)
plt.show()

In [ ]:
# A coeficent analasys
nfeatures = len(ridge_cv_model.coef_)
plt.figure(dpi = 800)
plt.barh(range(nfeatures), ridge_cv_model.coef_, align='center')
plt.yticks(np.arange(nfeatures), X.columns)
plt.xlabel("Feature importance")
plt.ylabel("Feature")